In [ ]:
!pip uninstall -y langchain langchain-core langchain-community langchain-google-genai google-generativeai pydantic chromadb pypdf gradio
!pip install --upgrade --quiet \
langchain \
langchain-google-genai \
google-generativeai \
gradio \
chromadb \
pypdf \
langchain_community

In [ ]:
!pip install unstructured "unstructured[pdf,txt,docx,csv,json,md]"

In [1]:
# Load Google Gemini (Generative AI) model
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

# LangChain core components
from langchain.chains import RetrievalQA
from langchain.chains.question_answering import load_qa_chain
from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate

# PDF reading and document loading
from langchain_community.document_loaders import UnstructuredFileLoader

# Vector store (Chroma for embeddings)
from langchain_community.vectorstores import Chroma

# Text splitting
from langchain.text_splitter import RecursiveCharacterTextSplitter

import gradio as gr
import os

In [2]:
from google.colab import drive
import os
from dotenv import load_dotenv

drive.mount('/content/drive')

# Assume .env is at /content/drive/MyDrive/.env
load_dotenv('/content/drive/MyDrive/GeminiAPI/.env')

# Now you can access it
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

Mounted at /content/drive


In [15]:
def get_llm():
    if "GOOGLE_API_KEY" not in os.environ:
        raise ValueError("GOOGLE_API_KEY not set. Please set it securely before calling get_llm().")

    model_id = "gemini-2.5-flash"

    parameters = {
        "temperature": 0.8,
        # "top_p": 1,
        # "top_k": 1,
        "max_output_tokens": 800000,
    }

    gemini_llm = ChatGoogleGenerativeAI(
        model=model_id,
        temperature=parameters["temperature"],
        # top_p=parameters["top_p"],
        # top_k=parameters["top_k"],
        max_output_tokens=parameters["max_output_tokens"]
    )

    return gemini_llm

In [16]:
def document_loader(file):
    loader = UnstructuredFileLoader(file.name)
    loaded_document = loader.load()
    return loaded_document

In [17]:
def text_splitter(data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=400,
        chunk_overlap=25,
        length_function=len,
    )
    chunks = text_splitter.split_documents(data)

    return chunks

In [18]:
def gemini_embedding():
    if "GOOGLE_API_KEY" not in os.environ:
        raise ValueError("GOOGLE_API_KEY not set. Please set it securely before calling gemini_embedding().")

    model_id = "models/embedding-001"

    embed_params = {
        # Gemini embedding API doesn't currently expose parameters like dimension, etc.
    }

    gemini_embedding_model = GoogleGenerativeAIEmbeddings(
        model=model_id
    )

    return gemini_embedding_model

In [19]:
def vector_database(chunks):
    embedding_model = gemini_embedding()
    vectordb = Chroma.from_documents(chunks , embedding_model)

    return vectordb

In [20]:
def retriever(file):
    splits = document_loader(file)
    chunks = text_splitter(splits)
    vectordb = vector_database(chunks)
    retriever = vectordb.as_retriever()

    return retriever

In [23]:
def chatbot_agent(query, file, history):
    if query == "":
        return history, history, None, ""

    llm = get_llm()
    if file is None:
        answer = llm.invoke(query).content

    else:
        results = retriever(file).vectorstore.similarity_search_with_score(query, k=4)

        threshold = 0.8  # adjust strictness; lower is more strict
        relevant_docs = [doc for doc, score in results if score < threshold]

        if relevant_docs:
            prompt_template = """
            Use the following pieces of context to answer the question.

            Context: {context}

            Question: {question}

            Helpful Answer:
            """
            PROMPT = PromptTemplate(
                template=prompt_template,
                input_variables=["context", "question"]
            )

            chain = load_qa_chain(llm, chain_type="stuff", prompt=PROMPT)

            answer = chain.invoke({
                "input_documents": relevant_docs,
                "question": query
            }, return_only_outputs=True)["output_text"]

        else:
            answer = llm.invoke(query).content

    if history is None:
        history = []
    history.append((query, answer))
    return history, history, None, ""

In [44]:
with gr.Blocks() as demo:
    gr.Markdown("# Chatbot with Document")

    chatbot = gr.Chatbot(label="Conversation", elem_id="chatbot")

    with gr.Row():
        query_input = gr.Textbox(
            label="Your Message",
            placeholder="Type a message here... 📎",
            lines=9
        )

        file_input = gr.File(
          label="",
          file_types=[".pdf", ".docx", ".txt", ".csv", ".md", ".json"],
          file_count="single",
          type="filepath",
          show_label=False,
          elem_id="file_icon"
        )

    state = gr.State([])

    query_input.submit(
        chatbot_agent,
        inputs=[query_input, file_input, state],
        outputs=[chatbot, state, file_input, query_input]
    )

/tmp/ipython-input-44-3443860924.py:4: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Conversation", elem_id="chatbot")


In [45]:
if __name__ == "__main__":
    demo.launch(share=False, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

Keyboard interruption in main thread... closing server.
